In [33]:
import numpy as np
import pandas as pd

In [34]:
logs=pd.read_excel(r'C:\Users\gayat\Desktop\new projects\Enterprise Database Performance Monitoring System\Dataset\query_logs.xlsx')

In [35]:
logs.head()
logs.tail()
logs.info()
logs.columns
logs.describe()
logs.head()

<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   log_id          2000 non-null   int64  
 1   user_id         1922 non-null   float64
 2   query_type      2000 non-null   str    
 3   execution_time  2000 non-null   float64
 4   cpu_usage       2000 non-null   float64
 5   memory_usage    1928 non-null   float64
 6   execution_date  2000 non-null   str    
 7   status          2000 non-null   str    
dtypes: float64(4), int64(1), str(3)
memory usage: 125.1 KB


,log_id,user_id,query_type,execution_time,cpu_usage,memory_usage,execution_date,status
0,1,1030.0,JOIN,2.1514,4.04,17.28,2026-03-18,TIMEOUT
1,2,862.0,ANALYZE,1.3741,25.70,47.25,2026-04-07,FAILED
2,3,904.0,INSERT,0.0175,1.65,7.30,2026-04-15,FAILED
3,4,1005.0,CREATE_INDEX,52.6362,99.70,96.84,2026-04-14,SUCCESS
4,5,1528.0,JOIN,0.2000,12.24,35.66,2026-02-07,TIMEOUT


In [36]:
#log_id
logs["log_id"].isnull().sum()
logs["log_id"].duplicated().sum()

np.int64(0)

In [37]:
users=pd.read_csv(r'C:\Users\gayat\Desktop\new projects\Enterprise Database Performance Monitoring System\Dataset\clean_users_dataset.csv')

In [38]:
#user_id
logs["user_id"].isnull().sum()
logs=logs.dropna(subset=["user_id"])
logs["user_id"].isnull().sum()

logs["user_id"].dtype
logs["user_id"]=logs["user_id"].astype(int)
logs["user_id"].dtype

invalid_ids=logs[~logs["user_id"].isin(users["user_id"])]
invalid_ids

logs=logs[logs["user_id"].isin(users["user_id"])]

logs[~logs["user_id"].isin(users["user_id"])]

logs[logs["user_id"] <= 0]

logs["user_id"].dtype





dtype('int64')

In [39]:
#query-type

logs["query_type"].unique()

logs["query_type"]=logs["query_type"].str.strip().str.upper()

logs["query_type"].unique()

logs["query_type"]= logs["query_type"].replace({
  "CREATE_INDEX":"CREATE INDEX"
})

logs["query_type"].unique()

<StringArray>
[        'JOIN',      'ANALYZE',       'INSERT', 'CREATE INDEX',
       'SELECT',       'DELETE',       'UPDATE',  'ALTER TABLE']
Length: 8, dtype: str

In [40]:
#execution-time
logs[logs["execution_time"]<0]
(logs["execution_time"]<0).sum()

logs=logs[logs["execution_time"] >= 0]
(logs["execution_time"]<0).sum()

logs["execution_time"].describe()

Q1=logs["execution_time"].quantile(0.25)
Q3=logs["execution_time"].quantile(0.75)

print(Q1)
print(Q3)

IQR=Q3-Q1
print(IQR)

lower_bound=Q1 - 1.5 * IQR
upper_bound=Q3 + 1.5 * IQR

print(lower_bound)
print(upper_bound)

outliers = logs[
  (logs["execution_time"] < lower_bound) | 
  (logs["execution_time"]> upper_bound)
              ]

print(outliers)

logs=logs[(logs["execution_time"] >= lower_bound) & 
          (logs["execution_time"] <= upper_bound)
        ]

logs["execution_time"].describe()


0.4597
1.9649
1.5052
-1.7981
4.2227
      log_id  user_id    query_type  execution_time  cpu_usage  memory_usage  \
3          4     1005  CREATE INDEX         52.6362      99.70         96.84   
12        13      887        INSERT         30.8349      87.25         95.73   
17        18      300        DELETE         70.6107      93.20         94.65   
29        30     1833        INSERT         46.7405      89.09         87.53   
42        43     1406        UPDATE        113.1359      90.84         97.26   
...      ...      ...           ...             ...        ...           ...   
1961    1962      851          JOIN       9999.9900     150.50           NaN   
1964    1965     1557        UPDATE         78.3852      88.71         80.46   
1966    1967     1221        DELETE        114.2780      97.61         94.74   
1974    1975      571       ANALYZE         35.0957      92.37         85.34   
1992    1993      860        INSERT       9999.9900     150.50           NaN   

   

count    1608.000000
mean        1.121162
std         0.767796
min         0.001600
25%         0.401475
50%         1.093200
75%         1.756300
max         2.497700
Name: execution_time, dtype: float64

In [41]:
#cpu_usage

logs["cpu_usage"].dtype
logs["cpu_usage"].isnull().sum()
(logs["cpu_usage"]<0).sum()
(logs["cpu_usage"]>100).sum()

logs["cpu_usage"].describe()


count    1608.000000
mean       15.779882
std        10.612267
min         0.100000
25%         5.995000
50%        15.270000
75%        24.812500
max        34.970000
Name: cpu_usage, dtype: float64

In [42]:
# memory_usage
logs["memory_usage"].dtype
logs["memory_usage"].isnull().sum()
(logs["memory_usage"] < 0).sum()
logs["memory_usage"].describe()

count    1608.000000
mean       29.461580
std        16.741815
min         1.080000
25%        14.317500
50%        28.375000
75%        44.125000
max        59.960000
Name: memory_usage, dtype: float64

In [43]:
# execution_date
logs["execution_date"].dtype
logs["execution_date"]=pd.to_datetime(
  logs["execution_date"],format="mixed",dayfirst=True,errors="coerce"
)
logs[logs["execution_date"].isnull()]
logs=logs.dropna(subset=["execution_date"])
logs["execution_date"].isnull().sum()
logs["execution_date"].dtype

dtype('<M8[us]')

In [44]:
# status
logs['status'].dtype
logs['status'].isnull().sum()
logs['status'].unique()
logs['status']=(logs["status"].str.strip().str.upper())
logs['status'].unique()

<StringArray>
['TIMEOUT', 'FAILED', 'SUCCESS', 'ERROR', 'PENDING']
Length: 5, dtype: str

In [45]:
logs.info()

logs.isnull().sum()

logs.duplicated().sum()

<class 'pandas.DataFrame'>
Index: 1601 entries, 0 to 1999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   log_id          1601 non-null   int64         
 1   user_id         1601 non-null   int64         
 2   query_type      1601 non-null   str           
 3   execution_time  1601 non-null   float64       
 4   cpu_usage       1601 non-null   float64       
 5   memory_usage    1601 non-null   float64       
 6   execution_date  1601 non-null   datetime64[us]
 7   status          1601 non-null   str           
dtypes: datetime64[us](1), float64(3), int64(2), str(2)
memory usage: 112.6 KB


np.int64(0)

In [46]:
logs.to_csv("clean_logs_dataset.csv", index=False)